# 04 PyTorch Model Evaluation v1

PyTorch evaluation report parity for GreenSpace CNN. This notebook mirrors the TensorFlow NB04 outputs for loss monitoring, threshold tuning, and split-wise reports.

In [1]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

os.environ.setdefault('MPLCONFIGDIR', '/private/tmp')

print('PROJECT_ROOT:', PROJECT_ROOT)

PROJECT_ROOT: /Users/starsrain/2025_codeProject/GreenSpace_CNN


In [2]:
import numpy as np
import pandas as pd
import torch

from src_torch.config import TORCH_DATA_CONFIG
from src_torch.data import load_split_dfs, resolve_split_schema
from src_torch.evaluation import (
    evaluate_all_splits,
    evaluate_loss_monitoring,
    find_latest_pytorch_checkpoint,
    infer_run_tag_and_variant,
    load_torch_checkpoint_model,
    predict_split,
    save_evaluation_outputs,
    tune_thresholds_f1,
)
from src_torch.training import resolve_device

split_dfs = load_split_dfs()
train_df = split_dfs['train']
val_df = split_dfs['val']
test_df = split_dfs['test']
schema = resolve_split_schema(train_df)
binary_cols = schema.binary_cols
bin_names = schema.bin_names

print('torch:', torch.__version__)
print('Loaded splits:', {k: len(v) for k, v in split_dfs.items()})
print('Binary labels:', binary_cols)

torch: 2.10.0
Loaded splits: {'train': 3067, 'val': 1022, 'test': 1022}
Binary labels: ['sports_field_p', 'multipurpose_open_area_p', 'children_s_playground_p', 'water_feature_p', 'walking_paths_p', 'built_structures_p', 'parking_lots_p']


## Load PyTorch Checkpoint

In [3]:
USE_MANUAL_MODEL = True
MANUAL_MODEL_PATH = PROJECT_ROOT / 'models/runs/PyTorch_20260608_220356/best_mcmae_PyTorch_20260608_220356.pt'
## /Users/starsrain/2025_codeProject/GreenSpace_CNN/models/runs/PyTorch_20260608_220356/best_mcmae_PyTorch_20260608_220356.pt
if USE_MANUAL_MODEL:
    model_path = Path(MANUAL_MODEL_PATH)
else:
    model_path = find_latest_pytorch_checkpoint(preferred_variant='best_mcmae')

assert model_path.exists(), f'Missing PyTorch checkpoint: {model_path}'
run_tag, EVAL_VARIANT = infer_run_tag_and_variant(model_path)
device = resolve_device('auto')
model, model_config, checkpoint = load_torch_checkpoint_model(model_path, device=device)

print('Loaded checkpoint:', model_path)
print('run_tag:', run_tag)
print('EVAL_VARIANT:', EVAL_VARIANT)
print('device:', device)

Loaded checkpoint: /Users/starsrain/2025_codeProject/GreenSpace_CNN/models/runs/PyTorch_20260608_220356/best_mcmae_PyTorch_20260608_220356.pt
run_tag: PyTorch_20260608_220356
EVAL_VARIANT: best_mcmae
device: mps


## Predict Splits

In [4]:
# Set to an integer for quick debug; use None for full split reports.
EVAL_MAX_BATCHES = None
EVAL_BATCH_SIZE = TORCH_DATA_CONFIG['batch_size']

predictions_by_split = {}
for split in ('train', 'val', 'test'):
    print(f'Predicting {split}...')
    predictions_by_split[split] = predict_split(
        model,
        split,
        device=device,
        batch_size=EVAL_BATCH_SIZE,
        max_batches=EVAL_MAX_BATCHES,
    )
    df_used, preds, loss_metrics = predictions_by_split[split]
    print(split, 'n=', len(df_used), 'loss=', round(loss_metrics['loss'], 4))

Predicting train...
train n= 3067 loss= 1.4547
Predicting val...
val n= 1022 loss= 2.2194
Predicting test...
test n= 1022 loss= 2.1776


## Loss Monitoring

In [5]:
loss_monitor_df = evaluate_loss_monitoring(predictions_by_split, binary_cols)
display(loss_monitor_df.round(4))

,split,loss,bin_head_loss,shade_head_loss,score_head_loss,veg_head_loss,bin_head_binary_accuracy,bin_head_pr_auc,shade_head_sparse_categorical_accuracy,score_head_mae,veg_head_mae
0,train,1.4547,0.2594,0.3615,0.4542,0.3796,0.8748,0.8833,0.8455,0.4542,0.3796
1,val,2.2194,0.2958,0.7721,0.6350,0.5164,0.8591,0.8423,0.6719,0.6350,0.5164
2,test,2.1776,0.2912,0.7757,0.6181,0.4927,0.8609,0.8465,0.6797,0.6181,0.4927


## Threshold Tuning

In [6]:
val_used_df, val_preds, _ = predictions_by_split['val']
hard_names_val = [name for name in bin_names if name in val_used_df.columns]
if hard_names_val:
    y_bin_val_true = val_used_df[hard_names_val].fillna(0).astype(int).values
    pred_bin_val_aligned = np.stack([val_preds['bin_head'][:, bin_names.index(name)] for name in hard_names_val], axis=1)
    label_names = hard_names_val
else:
    y_bin_val_true = (val_used_df[binary_cols].fillna(0.0).astype(np.float32).values >= 0.5).astype(int)
    pred_bin_val_aligned = val_preds['bin_head']
    label_names = bin_names

thresholds_df = tune_thresholds_f1(y_bin_val_true, pred_bin_val_aligned, label_names)
best_thresholds = {
    row['label']: float(row['best_threshold'])
    for _, row in thresholds_df.iterrows()
    if np.isfinite(row['best_threshold'])
}

print('Tuned thresholds:', len(best_thresholds), '/', len(label_names))
display(thresholds_df.sort_values('best_f1', ascending=False).reset_index(drop=True).round(4))

Tuned thresholds: 7 / 7


,label,best_threshold,best_f1,best_precision,best_recall,pos_rate,n_pos,n,note
0,multipurpose_open_area,0.3775,0.9403,0.9248,0.9563,0.8053,823,1022,
1,walking_paths,0.1755,0.9138,0.8612,0.9732,0.7671,784,1022,
2,sports_field,0.3747,0.8242,0.8536,0.7969,0.2505,256,1022,
3,built_structures,0.3198,0.8074,0.7751,0.8426,0.4041,413,1022,
4,parking_lots,0.3679,0.8034,0.8074,0.7993,0.2926,299,1022,
5,water_feature,0.2985,0.6907,0.7444,0.6442,0.2035,208,1022,
6,children_s_playground,0.3053,0.5645,0.4909,0.6639,0.1194,122,1022,


## Full Split Evaluation

In [7]:
overall_split_df, per_label_split_df = evaluate_all_splits(
    predictions_by_split,
    binary_cols,
    best_thresholds,
)

print('Overall metrics by split')
display(overall_split_df.round(4))
print('Per-label metrics by split')
display(per_label_split_df.round(4))

Overall metrics by split


,split,n_samples,n_labels,overall_F1@0.5,overall_ROC_AUC,overall_PR_AUC,overall_F1@tuned,shade_acc_overall,shade_acc_conditional,shade_acc_paths_present,score_acc,veg_acc,score_mae,veg_mae,score_ce,veg_ce,score_mae_mean,veg_mae_mean
0,train,3067,7,0.8062,0.9371,0.8833,0.8212,0.8455,0.8997,0.8963,0.6038,0.7101,0.5204,0.4016,NaN,NaN,0.4542,0.3796
1,val,1022,7,0.7615,0.9119,0.8423,0.7920,0.6712,0.6840,0.6798,0.4579,0.5685,0.6993,0.5403,NaN,NaN,0.6347,0.5164
2,test,1022,7,0.7782,0.9163,0.8465,0.7968,0.6800,0.6792,0.6764,0.4726,0.6174,0.6847,0.5063,NaN,NaN,0.6191,0.4932


Per-label metrics by split


,split,label,support_pos,support_neg,P@0.5,R@0.5,F1@0.5,ROC_AUC,PR_AUC,thr_val,P@thr,R@thr,F1@thr
0,train,multipurpose_open_area,2419,648,0.9373,0.9454,0.9413,0.9391,0.9797,0.3775,0.9283,0.9636,0.9456
1,train,walking_paths,2334,733,0.9213,0.9126,0.9169,0.9252,0.9720,0.1755,0.8705,0.9820,0.9229
2,train,sports_field,820,2247,0.9030,0.8512,0.8763,0.9745,0.9503,0.3747,0.8671,0.8915,0.8791
3,train,parking_lots,954,2113,0.8525,0.7694,0.8088,0.9532,0.9072,0.3679,0.8237,0.8375,0.8306
4,train,built_structures,1294,1773,0.8571,0.7743,0.8136,0.9280,0.9073,0.3198,0.7934,0.8609,0.8258
5,train,water_feature,615,2452,0.8753,0.5821,0.6992,0.9225,0.8148,0.2985,0.7731,0.7203,0.7458
6,train,children_s_playground,381,2686,0.6580,0.5302,0.5872,0.9175,0.6520,0.3053,0.5141,0.7165,0.5987
7,val,multipurpose_open_area,823,199,0.9311,0.9356,0.9333,0.9128,0.9739,0.3775,0.9248,0.9563,0.9403
8,val,walking_paths,784,238,0.9018,0.9018,0.9018,0.8912,0.9597,0.1755,0.8612,0.9732,0.9138
9,val,sports_field,256,766,0.8889,0.7500,0.8136,0.9614,0.9121,0.3747,0.8536,0.7969,0.8242


## Save Evaluation Artifacts

In [8]:
saved_paths = save_evaluation_outputs(
    run_tag=run_tag,
    variant=EVAL_VARIANT,
    loss_monitor_df=loss_monitor_df,
    thresholds_df=thresholds_df,
    overall_df=overall_split_df,
    per_label_df=per_label_split_df,
)

for name, path in saved_paths.items():
    print(name, '->', path)

loss_monitor -> /Users/starsrain/2025_codeProject/GreenSpace_CNN/monitoring_output/runs/PyTorch_20260608_220356/loss_monitor_best_mcmae.csv
thresholds -> /Users/starsrain/2025_codeProject/GreenSpace_CNN/monitoring_output/runs/PyTorch_20260608_220356/thresholds_best_mcmae.csv
overall -> /Users/starsrain/2025_codeProject/GreenSpace_CNN/report_outputs/runs/PyTorch_20260608_220356/overall_metrics_by_split_best_mcmae.csv
per_label -> /Users/starsrain/2025_codeProject/GreenSpace_CNN/report_outputs/runs/PyTorch_20260608_220356/per_label_metrics_by_split_best_mcmae.csv
